# ステップバイステップ・トレーニング・ノートブック

このノートブックでは、チェックポイントから学習済みモデルを読み込み、
**1バッチずつ**トレーニングを実行できます。

## 機能
- チェックポイントからモデル読み込み
- 1バッチ（1データサンプル群）ずつ実行
- トレーニング頻度の細かい調整
- リアルタイムでの可視化
- バッチごとの詳細なログ

## 1. セットアップ

In [ ]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import pandas as pd
from pathlib import Path
from IPython.display import clear_output, display
import time

# プロジェクトのモジュールをインポート
from train import get_command_line_parser, initialize_trainer
from utils import set_seed, set_gpu

# Jupyter用の設定
%matplotlib inline
%load_ext autoreload
%autoreload 2

plt.style.use('seaborn-v0_8-darkgrid')

print(f"Project root: {project_root}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. トレーナーの初期化

In [ ]:
# コマンドライン引数を設定
parser = get_command_line_parser()
args = parser.parse_args([])

# シードを設定
set_seed(args.seed)

# トレーナーを初期化
print("Initializing trainer...")
trainer = initialize_trainer(args)

print(f"\n📊 Trainer Configuration:")
print(f"  Dataset: {trainer.args.dataset}")
print(f"  Encoder: {trainer.args.encoder}")
print(f"  Project: {trainer.args.project}")
print(f"  Base classes: {trainer.args.base_class}")
print(f"  Total classes: {trainer.args.num_classes}")
print(f"  Way: {trainer.args.way}")
print(f"  Shot: {trainer.args.shot}")
print(f"  Total sessions: {trainer.args.sessions}")
print(f"  New sessions: {list(range(1, trainer.args.sessions))}")

## 3. チェックポイントから読み込み

In [ ]:
# 利用可能なチェックポイントを確認
checkpoint_dir = Path(trainer.args.save_path)

print(f"📁 Checkpoint directory: {checkpoint_dir}")
print(f"\nAvailable checkpoints:")

if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob('*.pth'))
    for i, cp in enumerate(checkpoints):
        size_mb = cp.stat().st_size / (1024 * 1024)
        print(f"  [{i}] {cp.name} ({size_mb:.2f} MB)")
else:
    print(f"  ⚠ Directory not found. Please run base session training first.")

In [ ]:
# ベースセッションのチェックポイントを読み込み
base_checkpoint = checkpoint_dir / 'session0_max_acc.pth'

if base_checkpoint.exists():
    print(f"📥 Loading checkpoint: {base_checkpoint.name}")
    checkpoint = torch.load(base_checkpoint)
    
    # モデルの状態を復元
    trainer.model.load_state_dict(checkpoint['params'])
    
    print(f"\n✅ Checkpoint loaded successfully!")
    print(f"  Session: {checkpoint.get('session', 0)}")
    print(f"  Max accuracy: {checkpoint.get('max_acc', 'N/A')}")
    
    # ベースセッションのテスト
    print(f"\n🧪 Testing base session...")
    _, test_acc = trainer.test(trainer.model, 0, return_loss=False)
    print(f"  Base session accuracy: {test_acc:.4f}")
else:
    print(f"❌ Error: Checkpoint not found: {base_checkpoint}")
    print(f"Please run base session training first:")
    print(f"  dvc repro train_base")

## 4. トレーニングパラメータの設定

In [ ]:
# トレーニングパラメータをカスタマイズ
training_config = {
    'target_session': 1,           # トレーニングするセッション
    'learning_rate': 0.1,          # 学習率
    'max_batches': 1000,           # 最大バッチ数
    'test_interval': 50,           # テスト実行間隔（バッチ）
    'log_interval': 10,            # ログ出力間隔（バッチ）
    'save_interval': 100,          # チェックポイント保存間隔
    'plot_update_interval': 20,    # グラフ更新間隔
}

print("⚙️ Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

## 5. データローダーの準備

In [ ]:
# 指定されたセッションのデータローダーを取得
session = training_config['target_session']

print(f"📦 Loading data for session {session}...")
trainset, trainloader, testloader = trainer.get_dataloader(session)

# データローダーのイテレータを作成
train_iter = iter(trainloader)
current_batch = 0
current_epoch = 0

print(f"\n📊 Dataset info:")
print(f"  Training samples: {len(trainset)}")
print(f"  Training batches: {len(trainloader)}")
print(f"  Test batches: {len(testloader)}")

# サンプルデータを確認
if hasattr(trainset, 'targets'):
    unique_classes = np.unique(trainset.targets)
    print(f"  Classes in this session: {unique_classes}")
    print(f"  Number of classes: {len(unique_classes)}")

## 6. トレーニング履歴の初期化

In [ ]:
# トレーニング履歴を記録
history = {
    'batch': [],
    'epoch': [],
    'loss': [],
    'accuracy': [],
    'test_acc': [],
    'learning_rate': [],
}

# 統計情報
stats = {
    'total_batches': 0,
    'total_samples': 0,
    'best_test_acc': 0.0,
    'best_batch': 0,
}

print("📝 Training history initialized")

## 7. 1バッチずつトレーニング（このセルを繰り返し実行）

**このセルを実行するたびに1バッチ分のトレーニングが進みます**

In [ ]:
# 1バッチのトレーニング
if stats['total_batches'] < training_config['max_batches']:
    try:
        # 次のバッチを取得
        batch = next(train_iter)
    except StopIteration:
        # エポック終了、新しいエポックを開始
        train_iter = iter(trainloader)
        batch = next(train_iter)
        current_epoch += 1
        print(f"\n🔄 Epoch {current_epoch} started")
    
    # データを取得
    if trainer.args.dataset == 'CICIDS2017_improved':
        data, target = batch
    else:
        data, target = batch[0], batch[1]
    
    data, target = data.cuda(), target.cuda()
    batch_size = data.size(0)
    
    # フォワードパス
    trainer.model.train()
    trainer.optimizer.zero_grad()
    output = trainer.model(data)
    
    # ロス計算
    loss = trainer.criterion(output, target)
    
    # バックワード
    loss.backward()
    trainer.optimizer.step()
    
    # 精度計算
    _, predicted = output.max(1)
    correct = predicted.eq(target).sum().item()
    accuracy = 100. * correct / batch_size
    
    # 統計更新
    stats['total_batches'] += 1
    stats['total_samples'] += batch_size
    current_batch += 1
    
    # 履歴に記録
    history['batch'].append(stats['total_batches'])
    history['epoch'].append(current_epoch)
    history['loss'].append(loss.item())
    history['accuracy'].append(accuracy)
    history['learning_rate'].append(trainer.optimizer.param_groups[0]['lr'])
    
    # ログ出力
    if stats['total_batches'] % training_config['log_interval'] == 0 or stats['total_batches'] == 1:
        print(f"\n📊 Batch {stats['total_batches']} (Epoch {current_epoch}):")
        print(f"  Loss: {loss.item():.4f}")
        print(f"  Accuracy: {accuracy:.2f}%")
        print(f"  Batch size: {batch_size}")
        print(f"  Total samples: {stats['total_samples']}")
    
    # テスト実行
    if stats['total_batches'] % training_config['test_interval'] == 0:
        print(f"\n🧪 Running test...")
        _, test_acc = trainer.test(trainer.model, session, return_loss=False)
        history['test_acc'].append(test_acc)
        
        if test_acc > stats['best_test_acc']:
            stats['best_test_acc'] = test_acc
            stats['best_batch'] = stats['total_batches']
            print(f"  ✨ New best test accuracy: {test_acc:.4f}")
        else:
            print(f"  Test accuracy: {test_acc:.4f} (best: {stats['best_test_acc']:.4f})")
    
    # 簡易進捗表示
    progress = (stats['total_batches'] / training_config['max_batches']) * 100
    print(f"\n⏳ Progress: {progress:.1f}% ({stats['total_batches']}/{training_config['max_batches']} batches)")
    
else:
    print("\n✅ Training completed! Maximum batches reached.")
    print(f"  Total batches: {stats['total_batches']}")
    print(f"  Total epochs: {current_epoch}")
    print(f"  Best test accuracy: {stats['best_test_acc']:.4f} (at batch {stats['best_batch']})")

## 8. リアルタイム可視化

In [ ]:
# トレーニング履歴を可視化
if len(history['batch']) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history['batch'], history['loss'], 'b-', alpha=0.6, linewidth=1)
    if len(history['loss']) > 10:
        # 移動平均を追加
        window = 10
        moving_avg = pd.Series(history['loss']).rolling(window=window).mean()
        axes[0, 0].plot(history['batch'], moving_avg, 'r-', linewidth=2, label=f'Moving Avg ({window})')
    axes[0, 0].set_xlabel('Batch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(history['batch'], history['accuracy'], 'g-', alpha=0.6, linewidth=1, label='Batch Acc')
    if len(history['accuracy']) > 10:
        moving_avg = pd.Series(history['accuracy']).rolling(window=10).mean()
        axes[0, 1].plot(history['batch'], moving_avg, 'r-', linewidth=2, label='Moving Avg (10)')
    axes[0, 1].set_xlabel('Batch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title('Training Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Test Accuracy
    if len(history['test_acc']) > 0:
        test_batches = [history['batch'][i] for i in range(0, len(history['batch']), training_config['test_interval']) if i < len(history['test_acc'])]
        test_batches = test_batches[:len(history['test_acc'])]
        axes[1, 0].plot(test_batches, [acc*100 for acc in history['test_acc']], 'ro-', linewidth=2, markersize=6)
        axes[1, 0].axhline(y=stats['best_test_acc']*100, color='g', linestyle='--', 
                          label=f"Best: {stats['best_test_acc']*100:.2f}%")
        axes[1, 0].set_xlabel('Batch')
        axes[1, 0].set_ylabel('Test Accuracy (%)')
        axes[1, 0].set_title('Test Accuracy')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'No test data yet', ha='center', va='center')
        axes[1, 0].set_title('Test Accuracy')
    
    # Learning Rate
    axes[1, 1].plot(history['batch'], history['learning_rate'], 'purple', linewidth=2)
    axes[1, 1].set_xlabel('Batch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_yscale('log')
    
    plt.tight_layout()
    plt.show()
    
    # 統計サマリー
    print(f"\n📈 Training Statistics:")
    print(f"  Total batches: {stats['total_batches']}")
    print(f"  Total epochs: {current_epoch}")
    print(f"  Total samples: {stats['total_samples']}")
    print(f"  Average loss (last 100): {np.mean(history['loss'][-100:]):.4f}")
    print(f"  Average accuracy (last 100): {np.mean(history['accuracy'][-100:]):.2f}%")
    if len(history['test_acc']) > 0:
        print(f"  Best test accuracy: {stats['best_test_acc']:.4f} (at batch {stats['best_batch']})")
        print(f"  Latest test accuracy: {history['test_acc'][-1]:.4f}")
else:
    print("No training history yet. Run the training cell above.")

## 9. 自動トレーニング（N バッチ実行）

複数バッチを自動的に実行したい場合は、このセルを使用します。

In [ ]:
# 自動実行の設定
auto_config = {
    'num_batches': 100,  # 実行するバッチ数
    'show_progress': True,  # プログレスバー表示
}

print(f"🚀 Running {auto_config['num_batches']} batches automatically...\n")

if auto_config['show_progress']:
    pbar = tqdm(range(auto_config['num_batches']), desc='Auto Training')
else:
    pbar = range(auto_config['num_batches'])

for _ in pbar:
    if stats['total_batches'] >= training_config['max_batches']:
        print("Maximum batches reached!")
        break
    
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(trainloader)
        batch = next(train_iter)
        current_epoch += 1
    
    # データを取得
    if trainer.args.dataset == 'CICIDS2017_improved':
        data, target = batch
    else:
        data, target = batch[0], batch[1]
    
    data, target = data.cuda(), target.cuda()
    batch_size = data.size(0)
    
    # トレーニング
    trainer.model.train()
    trainer.optimizer.zero_grad()
    output = trainer.model(data)
    loss = trainer.criterion(output, target)
    loss.backward()
    trainer.optimizer.step()
    
    # 精度計算
    _, predicted = output.max(1)
    correct = predicted.eq(target).sum().item()
    accuracy = 100. * correct / batch_size
    
    # 統計更新
    stats['total_batches'] += 1
    stats['total_samples'] += batch_size
    
    # 履歴に記録
    history['batch'].append(stats['total_batches'])
    history['epoch'].append(current_epoch)
    history['loss'].append(loss.item())
    history['accuracy'].append(accuracy)
    history['learning_rate'].append(trainer.optimizer.param_groups[0]['lr'])
    
    # テスト実行
    if stats['total_batches'] % training_config['test_interval'] == 0:
        _, test_acc = trainer.test(trainer.model, session, return_loss=False)
        history['test_acc'].append(test_acc)
        
        if test_acc > stats['best_test_acc']:
            stats['best_test_acc'] = test_acc
            stats['best_batch'] = stats['total_batches']
    
    # プログレスバーの更新
    if auto_config['show_progress']:
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'acc': f"{accuracy:.1f}%",
            'best_test': f"{stats['best_test_acc']:.4f}"
        })

print(f"\n✅ Auto training completed!")
print(f"  Batches processed: {auto_config['num_batches']}")
print(f"  Total batches: {stats['total_batches']}")
print(f"  Best test accuracy: {stats['best_test_acc']:.4f}")

## 10. チェックポイントの保存

In [ ]:
# 現在のモデルをチェックポイントとして保存
save_path = checkpoint_dir / f'session{session}_notebook_batch{stats["total_batches"]}.pth'

checkpoint = {
    'session': session,
    'batch': stats['total_batches'],
    'epoch': current_epoch,
    'params': trainer.model.state_dict(),
    'optimizer': trainer.optimizer.state_dict(),
    'max_acc': stats['best_test_acc'],
    'history': history,
    'stats': stats,
}

torch.save(checkpoint, save_path)
print(f"💾 Checkpoint saved: {save_path.name}")
print(f"  Session: {session}")
print(f"  Batches: {stats['total_batches']}")
print(f"  Epochs: {current_epoch}")
print(f"  Best test accuracy: {stats['best_test_acc']:.4f}")

## 11. 詳細な履歴分析

In [ ]:
# トレーニング履歴をDataFrameに変換
if len(history['batch']) > 0:
    df_history = pd.DataFrame({
        'batch': history['batch'],
        'epoch': history['epoch'],
        'loss': history['loss'],
        'accuracy': history['accuracy'],
        'lr': history['learning_rate'],
    })
    
    print("📊 Training History (last 20 batches):")
    print(df_history.tail(20).to_string(index=False))
    
    print(f"\n📈 Summary Statistics:")
    print(df_history[['loss', 'accuracy']].describe())
    
    # エポックごとの統計
    if current_epoch > 0:
        print(f"\n📅 Epoch-wise Statistics:")
        epoch_stats = df_history.groupby('epoch').agg({
            'loss': ['mean', 'min', 'max'],
            'accuracy': ['mean', 'min', 'max'],
            'batch': 'count'
        })
        print(epoch_stats)

## 12. 現在のバッチの詳細情報

In [ ]:
# 最新のバッチ情報を表示
if len(history['batch']) > 0:
    print(f"🔍 Current Batch Information:")
    print(f"  Batch number: {stats['total_batches']}")
    print(f"  Current epoch: {current_epoch}")
    print(f"  Total samples processed: {stats['total_samples']}")
    print(f"  Latest loss: {history['loss'][-1]:.4f}")
    print(f"  Latest accuracy: {history['accuracy'][-1]:.2f}%")
    print(f"  Current learning rate: {history['learning_rate'][-1]:.6f}")
    
    # 最近のトレンド
    if len(history['loss']) >= 10:
        recent_loss_trend = np.mean(history['loss'][-10:]) - np.mean(history['loss'][-20:-10]) if len(history['loss']) >= 20 else 0
        trend_symbol = "📉" if recent_loss_trend < 0 else "📈"
        print(f"\n  Loss trend (last 10 vs previous 10): {trend_symbol} {recent_loss_trend:+.4f}")
    
    # 進捗
    progress_pct = (stats['total_batches'] / training_config['max_batches']) * 100
    remaining = training_config['max_batches'] - stats['total_batches']
    print(f"\n  Progress: {progress_pct:.1f}% ({remaining} batches remaining)")